# Titanic — Model Inference

Load the best model from MLflow Model Registry and generate `submission.csv`.

In [ ]:
!pip install mlflow category_encoders dagshub --quiet

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import category_encoders as ce

warnings.filterwarnings('ignore')

## MLflow / DagsHub Setup

Must match the settings used in `model_experiment.ipynb`.

In [ ]:
USE_DAGSHUB = True

DAGSHUB_USERNAME = "YOUR_DAGSHUB_USERNAME"
DAGSHUB_REPO     = "YOUR_REPO_NAME"
DAGSHUB_TOKEN    = "YOUR_DAGSHUB_TOKEN"

if USE_DAGSHUB:
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
else:
    mlflow.set_tracking_uri("mlruns")

print("Tracking URI:", mlflow.get_tracking_uri())

## Load Model from Registry

In [ ]:
MODEL_NAME    = "titanic-best-model"
MODEL_VERSION = "latest"   # or a specific version number like "1"

model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
model     = mlflow.sklearn.load_model(model_uri)
print("Model loaded:", type(model).__name__)

## Load Preprocessing Artifacts

In [ ]:
with open('preprocessing_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_encoding    = artifacts['best_encoding']
selected_ohe     = artifacts['selected_ohe']
selected_woe     = artifacts['selected_woe']
woe_encoder      = artifacts['woe_encoder']
NUMERIC_COLS     = artifacts['numeric_cols']
CAT_COLS         = artifacts['cat_cols']
WOE_COLS         = artifacts['woe_cols']
OHE_SOURCE_COLS  = artifacts['ohe_source_cols']

print("Best encoding used:", best_encoding)

## Preprocessing Pipeline (mirrors model_experiment.ipynb)

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.drop(columns=['PassengerId'], errors='ignore', inplace=True)
    df.drop_duplicates(inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)
    return df.reset_index(drop=True)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Pclass']   = (df[['Pclass_1','Pclass_2','Pclass_3']]
                      .idxmax(axis=1)
                      .str.replace('Pclass_', '', regex=False)
                      .astype(int))
    df['Title']    = (df[['Title_1','Title_2','Title_3','Title_4']]
                      .idxmax(axis=1)
                      .str.replace('Title_', '', regex=False)
                      .astype(int))
    df['Embarked'] = (df[['Emb_1','Emb_2','Emb_3']]
                      .idxmax(axis=1)
                      .str.replace('Emb_', '', regex=False)
                      .astype(int))
    df['IsAlone']  = (df['Family_size'] == 0).astype(int)
    df['AgeBin']   = pd.qcut(df['Age'],  q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
    df['FareBin']  = pd.qcut(df['Fare'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')
    df.drop(columns=OHE_SOURCE_COLS, inplace=True)
    return df

## Load Test Data & Generate Submission

In [ ]:
test_raw      = pd.read_csv("data/test_data.csv", index_col=0)
passenger_ids = test_raw['PassengerId'].copy() if 'PassengerId' in test_raw.columns else test_raw.index

test_clean = clean(test_raw)
test_fe    = add_features(test_clean)

if best_encoding == 'OHE':
    ohe_dummies = pd.get_dummies(test_fe[CAT_COLS].astype(str), columns=CAT_COLS, drop_first=False)
    X_test = pd.concat([test_fe[NUMERIC_COLS].reset_index(drop=True), ohe_dummies], axis=1)
    for col in selected_ohe:
        if col not in X_test.columns:
            X_test[col] = 0
    X_test = X_test[selected_ohe]

else:
    def prep_woe(df):
        out = df[NUMERIC_COLS + WOE_COLS].copy()
        for c in ['AgeBin', 'FareBin']:
            out[c] = out[c].astype(str)
        return out.reset_index(drop=True)

    X_test = woe_encoder.transform(prep_woe(test_fe))[selected_woe]

print("Test feature matrix shape:", X_test.shape)
predictions = model.predict(X_test.values.astype(float))

submission = pd.DataFrame({
    'PassengerId': passenger_ids.values,
    'Survived':    predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv saved.")
submission.head(10)

In [ ]:
print("Survival rate in submission:", submission['Survived'].mean().round(4))
submission['Survived'].value_counts()